# 10 — Policy Scenario Comparison

Runs NSGA-II evolution under 5 different policy scenarios and compares:
- Pareto fronts (all overlaid)
- Action maps per scenario
- Summary statistics

Uses the learned carbon model with pre-computed forest GBR predictions.

**Expected runtime:** ~40-60 min total (5 scenarios × 200 pop × 200 gen)

In [ ]:
import sys
sys.path.insert(0, "../src")

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import time
from estonia_landuse.optimizer.trainer import train as run_evolution
from estonia_landuse.scenarios import build_scenario_summary, select_scenario_representatives
from estonia_landuse.simulator.config import default_config
from estonia_landuse.simulator.simulator import summarize_policy
from estonia_landuse.simulator.targets import realize_targets

## Configuration

In [ ]:
POP_SIZE = 200
N_GENERATIONS = 200

DATA_DIR = Path("../data/processed/learned_carbon")
RESULTS_DIR = DATA_DIR

## Load data

All features pre-computed by notebook 09 (includes forest, peat, rohemeeter).

In [ ]:
features_df = pd.read_parquet(DATA_DIR / "features_with_forest.parquet")

if "predicted_tco2_ha_yr" not in features_df.columns:
    raise FileNotFoundError(
        "Missing prepared predicted_tco2_ha_yr. "
        "Run notebook 09_spatial_join_and_model.ipynb first."
    )

valid_carbon = features_df["predicted_tco2_ha_yr"].dropna()
print(
    "Prepared cell carbon predictions: "
    f"{len(valid_carbon):,}/{len(features_df):,} cells, "
    f"mean={valid_carbon.mean():.2f} tCO2/ha/yr"
)

# Feature columns for prescriptor
FEATURE_COLUMNS = [
    "urban_pct", "agriculture_pct", "grassland_pct", "forest_pct",
    "wetland_pct", "water_pct", "naturalness_score", "carbon_score",
    "protected_overlap_pct", "wetland_suitability", "biodiversity_proxy",
    "opportunity_cost_proxy", "rohemeeter_norm",
]
FEATURE_COLUMNS = [c for c in FEATURE_COLUMNS if c in features_df.columns]

# Grid for maps
grid = gpd.read_file("../data/processed/v1/base_grid.gpkg")
print(f"Loaded: {len(features_df)} cells, {len(FEATURE_COLUMNS)} features")
print(f"  peat_overlap_pct: mean={features_df['peat_overlap_pct'].mean():.3f}, >0: {(features_df['peat_overlap_pct']>0).sum()}")
print(f"  rohemeeter_norm: mean={features_df.get('rohemeeter_norm', pd.Series(0)).mean():.3f}")

## Define scenarios

In [ ]:
def make_scenario_config(scenario_name):
    """Create a config dict for each scenario with aggressive differentiation."""
    config = default_config()
    config["carbon_model"] = "learned"

    if scenario_name == "green_maximum":
        # Maximum ecological gain — relax all economic constraints
        config["scoring"]["agriculture_loss_cost"] = 0.3
        config["max_total_agri_loss_pct"] = 0.50
        config["max_changed_pct"] = 0.40
        config["budget_penalty_weight"] = 3.0
        config["total_agri_loss_penalty_weight"] = 5.0

    elif scenario_name == "food_security":
        # Preserve farmland at all costs
        config["scoring"]["agriculture_loss_cost"] = 15.0
        config["max_total_agri_loss_pct"] = 0.03
        config["total_agri_loss_penalty_weight"] = 100.0
        config["max_changed_pct"] = 0.15

    elif scenario_name == "low_budget":
        # Minimal intervention — what can you achieve cheaply?
        config["max_changed_pct"] = 0.06
        config["max_total_agri_loss_pct"] = 0.15
        config["budget_penalty_weight"] = 50.0
        config["scoring"]["base_change_cost"] = 2.0

    elif scenario_name == "wetland_priority":
        # Prioritize wetland restoration — relax constraints on wetland
        config["constraints"]["wetland_suit_min_for_restore"] = 0.05
        config["scoring"]["biodiversity_value"] = [0.4, 1.0, 0.1, 0.3]
        config["max_changed_pct"] = 0.25
        config["max_total_agri_loss_pct"] = 0.15
        config["max_total_agri_gain_pct"] = 0.05
        config["max_gross_agri_gain_pct"] = 0.15
        config["scoring"]["agriculture_gain_cost"] = 10.0
        config["optimization"] = {}
        config["optimization"]["fourth_objective"] = "wetland_gain_pct"
        config["budget_penalty_weight"] = 5.0

    elif scenario_name == "sustainable_agriculture":
        # Expand agriculture within ecological and relocation safeguards
        config["max_changed_pct"] = 0.15
        config["max_total_agri_loss_pct"] = 1.0
        config["min_total_agri_gain_pct"] = 0.05
        config["max_total_agri_gain_pct"] = 0.10
        config["max_gross_agri_loss_pct"] = 0.02
        config["min_biodiversity_gain"] = -0.01
        config["min_carbon_gain"] = -0.01
        config["scoring"]["agriculture_loss_cost"] = 30.0
        config["scoring"]["agriculture_gain_cost"] = 0.0
        config["optimization"] = {}
        config["optimization"]["fourth_objective"] = "agriculture_gain_pct"

    elif scenario_name == "balanced":
        config["max_changed_pct"] = 0.20
        config["max_total_agri_loss_pct"] = 0.15

    return config


SCENARIOS = {
    "green_maximum": "Green Maximum (low agri protection)",
    "food_security": "Food Security (preserve farmland)",
    "low_budget": "Low Budget (minimal intervention)",
    "wetland_priority": "Wetland Priority (rewetting focus)",
    "sustainable_agriculture": "Sustainable Agriculture Expansion",
    "balanced": "Balanced (default)",
}
SELECTION_RULES = {
    "green_maximum": "green_maximum",
    "food_security": "food_security",
    "low_budget": "low_budget",
    "wetland_priority": "wetland_priority",
    "sustainable_agriculture": "sustainable_agriculture",
    "balanced": "balanced",
}

print("Scenarios defined:")
for key, desc in SCENARIOS.items():
    print(f"  {key}: {desc}")

## Run evolution for each scenario

In [ ]:
results = {}
for scenario_name, description in SCENARIOS.items():
    config = make_scenario_config(scenario_name)

    print(f"\n{'='*60}")
    print(f"Scenario: {description}")
    print(f"  pop={POP_SIZE}, gen={N_GENERATIONS}")
    t0 = time.time()

    pop = run_evolution(
        context=features_df,
        feature_columns=FEATURE_COLUMNS,
        pop_size=POP_SIZE,
        n_generations=N_GENERATIONS,
        config=config,
        seed=42,
        verbose=True,
    )

    dt = time.time() - t0
    front0 = [p for p in pop if p.rank == 0]
    results[scenario_name] = {
        "pop": pop, "front0": front0, "config": config,
        "time": dt, "description": description,
    }
    print(f"  Done in {dt:.0f}s, front-0: {len(front0)}")

print(f"\n\nTotal time: {sum(r['time'] for r in results.values())/60:.1f} min")

## Evaluate Pareto fronts

In [ ]:
# Evaluate each scenario's front with its own config
def get_pareto_df(population, context, config):
    features = context[FEATURE_COLUMNS].values.astype(np.float32)
    feat_mean = features.mean(axis=0)
    feat_std = features.std(axis=0)
    feat_std[feat_std == 0] = 1.0
    feat_norm = (features - feat_mean) / feat_std
    rows = []
    for i, p in enumerate(population):
        if p.rank != 0:
            continue
        targets = p.prescribe(feat_norm)
        s = summarize_policy(context, targets, config)
        s["id"] = i
        rows.append(s)
    return pd.DataFrame(rows), feat_norm

pareto_dfs = {}
for name, data in results.items():
    pdf, feat_norm = get_pareto_df(data["pop"], features_df, data["config"])
    pareto_dfs[name] = pdf
    print(f"{name}: {len(pdf)} front-0 policies")

## All Pareto Fronts (overlaid)

In [ ]:
colors = {
    "green_maximum": "tab:green",
    "food_security": "tab:red",
    "low_budget": "tab:gray",
    "wetland_priority": "tab:blue",
    "sustainable_agriculture": "tab:purple",
    "balanced": "tab:orange",
}

markers = {
    "green_maximum": "o",
    "food_security": "s",
    "low_budget": "D",
    "wetland_priority": "^",
    "sustainable_agriculture": "P",
    "balanced": "v",
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Bio vs Carbon
ax = axes[0, 0]
for name, pdf in pareto_dfs.items():
    ax.scatter(pdf["carbon_gain"], pdf["biodiversity_gain"],
               alpha=0.6, s=25, label=SCENARIOS[name],
               color=colors[name], marker=markers[name])
ax.set_xlabel("Carbon Gain"); ax.set_ylabel("Biodiversity Gain")
ax.set_title("Biodiversity vs Carbon"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Bio vs Cost
ax = axes[0, 1]
for name, pdf in pareto_dfs.items():
    ax.scatter(pdf["cost"], pdf["biodiversity_gain"],
               alpha=0.6, s=25, label=SCENARIOS[name],
               color=colors[name], marker=markers[name])
ax.set_xlabel("Cost"); ax.set_ylabel("Biodiversity Gain")
ax.set_title("Biodiversity vs Cost"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Carbon vs Cost
ax = axes[1, 0]
for name, pdf in pareto_dfs.items():
    ax.scatter(pdf["cost"], pdf["carbon_gain"],
               alpha=0.6, s=25, label=SCENARIOS[name],
               color=colors[name], marker=markers[name])
ax.set_xlabel("Cost"); ax.set_ylabel("Carbon Gain")
ax.set_title("Carbon vs Cost"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Bio vs Change%
ax = axes[1, 1]
for name, pdf in pareto_dfs.items():
    ax.scatter(pdf["changed_pct"], pdf["biodiversity_gain"],
               alpha=0.6, s=25, label=SCENARIOS[name],
               color=colors[name], marker=markers[name])
ax.set_xlabel("Changed %"); ax.set_ylabel("Biodiversity Gain")
ax.set_title("Biodiversity vs Land Change"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle("Pareto Fronts Across Policy Scenarios", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Summary table

In [ ]:
representatives = select_scenario_representatives(
    pareto_dfs,
    SELECTION_RULES,
)
selected_policies = {
    name: results[name]["pop"][int(row["id"])]
    for name, row in representatives.items()
}
summary = build_scenario_summary(
    pareto_dfs,
    representatives=representatives,
    selection_rules=SELECTION_RULES,
    scenario_labels=SCENARIOS,
    elapsed_seconds={name: data["time"] for name, data in results.items()},
)
print(summary.set_index("Scenario").to_string(float_format="{:.3f}".format))

In [ ]:
summary.round(3)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

metrics = [
    "Biodiversity gain",
    "Carbon gain",
    "Changed land",
    "Agriculture loss",
    "Agriculture gain",
    "Gross agriculture gain",
    "Wetland gain",
]

plot_df = summary[metrics].copy() * 100

# Use the Scenario column when summary has an integer index
if "Scenario" in summary.columns:
    scenario_labels = summary["Scenario"].astype(str)
else:
    scenario_labels = pd.Series(summary.index.astype(str))

plot_df.index = (
    scenario_labels
    .str.replace(r" \(.*\)", "", regex=True)
    .to_numpy()
)

axes = plot_df.plot(
    kind="bar",
    subplots=True,
    layout=(4, 2),
    figsize=(16, 16),
    legend=False,
    sharex=True,
    color="steelblue",
)

for ax, metric in zip(axes.flat, metrics):
    ax.set_title(metric)
    ax.set_ylabel("%")
    ax.grid(axis="y", alpha=0.25)

    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", fontsize=8)

for ax in axes.flat[len(metrics):]:
    ax.set_visible(False)

plt.suptitle("Scenario Representative Results", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## Action maps (scenario representative per scenario)

In [ ]:
# Compute best policy targets for each scenario
groups = ["forest", "wetland", "agriculture", "grassland"]

features = features_df[FEATURE_COLUMNS].values.astype(np.float32)
feat_mean = features.mean(axis=0)
feat_std = features.std(axis=0)
feat_std[feat_std == 0] = 1.0
feat_norm = (features - feat_mean) / feat_std

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes_flat = axes.flatten()

for idx, (name, data) in enumerate(results.items()):
    if idx >= 6:
        break
    ax = axes_flat[idx]

    # Reuse the scenario representative selected for the summary
    best_policy = selected_policies[name]

    # Compute delta
    targets = best_policy.prescribe(feat_norm)
    current = np.column_stack([features_df[f"{g}_pct"].values for g in groups])
    targets_norm = realize_targets(features_df, targets, data["config"])
    delta = targets_norm - current

    # Dominant action
    change_intensity = np.abs(delta).sum(axis=1) / 2.0
    dominant = np.array(groups)[delta.argmax(axis=1)]
    dominant[change_intensity < 0.05] = "no_change"

    # Map
    map_df = grid.copy()
    map_df["action"] = dominant

    color_map = {"forest": "#228B22", "wetland": "#4682B4",
                 "agriculture": "#DAA520", "grassland": "#90EE90", "no_change": "#D3D3D3"}
    for action, color in color_map.items():
        subset = map_df[map_df["action"] == action]
        if len(subset) > 0:
            subset.plot(ax=ax, color=color, edgecolor="none")

    ax.set_title(f"{SCENARIOS[name]}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

# Hide unused subplot
if len(results) < 6:
    axes_flat[5].set_visible(False)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=a) for a, c in color_map.items()]
fig.legend(handles=legend_elements, loc="lower right", fontsize=10, ncol=5)

plt.suptitle("Scenario Representative Map (dominant action)", fontsize=14)
plt.tight_layout()
plt.show()

## Change intensity comparison

In [ ]:
from matplotlib.colors import PowerNorm

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes_flat = axes.flatten()

for idx, (name, data) in enumerate(results.items()):
    if idx >= 6:
        break
    ax = axes_flat[idx]

    # Reuse the scenario representative selected for the summary
    best_policy = selected_policies[name]

    targets = best_policy.prescribe(feat_norm)
    current = np.column_stack([features_df[f"{g}_pct"].values for g in groups])
    targets_norm = realize_targets(features_df, targets, data["config"])
    delta = targets_norm - current

    intensity = np.abs(delta).sum(axis=1) / 2.0
    map_df = grid.copy()
    map_df["intensity"] = intensity

    map_df.plot(column="intensity", ax=ax, cmap="YlOrRd",
                norm=PowerNorm(gamma=0.4, vmin=0, vmax=0.5),
                legend=(idx == 0))
    ax.set_title(f"{SCENARIOS[name]}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

if len(results) < 6:
    axes_flat[5].set_visible(False)

plt.suptitle("Scenario Representative Change Intensity (power scale)", fontsize=14)
plt.tight_layout()
plt.show()

## Save results

In [ ]:
# Save pareto data for later analysis
all_pareto = []
for name, pdf in pareto_dfs.items():
    pdf_copy = pdf.copy()
    pdf_copy["scenario"] = name
    pdf_copy["is_representative"] = (
        pdf_copy["id"] == int(representatives[name]["id"])
    )
    all_pareto.append(pdf_copy)

pd.concat(all_pareto).to_parquet(RESULTS_DIR / "scenario_comparison.parquet", index=False)
summary.to_parquet(RESULTS_DIR / "scenario_summary.parquet", index=False)
print(f"Saved Pareto metrics to {RESULTS_DIR / 'scenario_comparison.parquet'}")
print(f"Saved scenario summary to {RESULTS_DIR / 'scenario_summary.parquet'}")

# Save per-cell action maps as GeoPackage (for interactive visualizations)
maps_dir = RESULTS_DIR / "scenario_maps"
maps_dir.mkdir(exist_ok=True)

for name, data in results.items():
    best_policy = selected_policies[name]

    targets = best_policy.prescribe(feat_norm)
    current = np.column_stack([features_df[f"{g}_pct"].values for g in groups])
    targets_norm = realize_targets(features_df, targets, data["config"])
    delta = targets_norm - current

    # Build map dataframe
    map_gdf = grid[["cell_id", "geometry"]].copy()
    map_gdf["change_intensity"] = np.abs(delta).sum(axis=1) / 2.0
    dominant = np.array(groups)[delta.argmax(axis=1)]
    dominant[map_gdf["change_intensity"].values < 0.05] = "no_change"
    map_gdf["action"] = dominant
    for i, g in enumerate(groups):
        map_gdf[f"current_{g}"] = current[:, i]
        map_gdf[f"target_{g}"] = targets_norm[:, i]
        map_gdf[f"delta_{g}"] = delta[:, i]

    map_gdf.to_file(maps_dir / f"{name}.gpkg", driver="GPKG")

print(f"Saved {len(results)} scenario maps to {maps_dir}/")
print("Each contains: cell_id, geometry, action, change_intensity, current/target/delta per group")

In [ ]:
# Show wetland gain directly; dominant-action maps can hide rewetting.
selected_wetland_maps = {}
shared_wetland_max = 0.0
for name, data in results.items():
    best_policy = selected_policies[name]
    targets = best_policy.prescribe(feat_norm)
    current = np.column_stack(
        [features_df[f"{group}_pct"].values for group in groups]
    )
    targets_norm = realize_targets(
        features_df, targets, data["config"]
    )
    delta = targets_norm - current
    map_df = grid.copy()
    map_df["wetland_gain"] = np.clip(delta[:, 1], 0, None)
    selected_wetland_maps[name] = map_df
    shared_wetland_max = max(
        shared_wetland_max, float(map_df["wetland_gain"].max())
    )

shared_wetland_max = max(shared_wetland_max, 1e-12)
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes_flat = axes.flatten()
for idx, name in enumerate(results):
    ax = axes_flat[idx]
    selected_wetland_maps[name].plot(
        column="wetland_gain",
        ax=ax,
        cmap="Blues",
        vmin=0,
        vmax=shared_wetland_max,
        legend=(idx == 0),
    )
    ax.set_title(SCENARIOS[name], fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
if len(results) < len(axes_flat):
    axes_flat[-1].set_visible(False)
plt.suptitle("Wetland Gain per Scenario Representative", fontsize=14)
plt.tight_layout()
plt.show()